In [59]:
import pandas as pd
import os
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

DARK_BG        = PatternFill(start_color="1E1E2E", end_color="1E1E2E", fill_type="solid")
HEADER_FILL    = PatternFill(start_color="2E2E4E", end_color="2E2E4E", fill_type="solid")
NO_FILL        = PatternFill(fill_type=None)
FONT_LIGHT     = Font(color="D4D4D4")
FONT_HEADER    = Font(color="FFFFFF", bold=True)
FONT_DARK      = Font(color="1E1E2E")
FONT_RED       = Font(color="CC3333")
FONT_RED_LIGHT = Font(color="FF6B6B")
THIN = Side(style="thin", color="3A3A5C")
GRID = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

# Order matters: more specific keys first (e.g. "datetime" before "int")
DTYPE_FILLS = {
    "datetime": PatternFill(start_color="3A1A5C", end_color="3A1A5C", fill_type="solid"),  # purple
    "float":    PatternFill(start_color="1A3A6E", end_color="1A3A6E", fill_type="solid"),  # blue
    "int":      PatternFill(start_color="004D40", end_color="004D40", fill_type="solid"),  # teal
    "bool":     PatternFill(start_color="3A4A1A", end_color="3A4A1A", fill_type="solid"),  # olive green
    "category": PatternFill(start_color="5C1A3A", end_color="5C1A3A", fill_type="solid"),  # rose
    "object":   PatternFill(start_color="5C3A00", end_color="5C3A00", fill_type="solid"),  # amber
}


In [60]:
Path = "Features build/Feature_data"

In [61]:
def check_file(filepath):
    df = pd.read_parquet(filepath)

    # 1. Column data types
    dtypes = df.dtypes.rename("dtype").reset_index().rename(columns={"index": "column"})

    # 2. Null counts / % per column broken down by year (vectorised)
    if isinstance(df.index, pd.DatetimeIndex):
        years = df.index.year.rename("year")
        null_counts = df.isnull().groupby(years).sum()
        year_totals = years.value_counts().sort_index()
        null_pct = null_counts.div(year_totals, axis=0).round(4)

        counts_by_year = null_counts.T
        counts_by_year.columns = [f"{yr}" for yr in counts_by_year.columns]
        counts_by_year.insert(0, "column", counts_by_year.index)
        counts_by_year["total"] = df.isnull().sum().values
        blank_counts = counts_by_year.reset_index(drop=True)

        pct_by_year = null_pct.T
        pct_by_year.columns = [f"{yr}" for yr in pct_by_year.columns]
        pct_by_year.insert(0, "column", pct_by_year.index)
        pct_by_year["total"] = (df.isnull().sum() / len(df)).round(4).values
        blank_pcts = pct_by_year.reset_index(drop=True)
    else:
        null_counts = df.isnull().sum()
        null_pct = (null_counts / len(df)).round(4)
        blank_counts = pd.DataFrame({"column": null_counts.index, "total": null_counts.values})
        blank_pcts = pd.DataFrame({"column": null_pct.index, "total": null_pct.values})

    # 3. Index date range and completeness (5-min intervals)
    idx = df.index
    if isinstance(idx, pd.DatetimeIndex):
        start, end = idx.min(), idx.max()
        expected = pd.date_range(start=start, end=end, freq="5min")
        missing = expected.difference(idx)
        date_info = pd.DataFrame({
            "start": [start], "end": [end],
            "expected_rows": [len(expected)], "actual_rows": [len(df)],
            "missing_intervals": [len(missing)]
        })
    else:
        date_info = pd.DataFrame([["Index is not a DatetimeIndex"]], columns=[""])

    return dtypes, blank_counts, blank_pcts, date_info


def autofit_columns(worksheet):
    for col in worksheet.columns:
        max_len = max((len(str(cell.value)) if cell.value is not None else 0) for cell in col)
        worksheet.column_dimensions[col[0].column_letter].width = max_len + 4


def apply_dark_theme(worksheet):
    headers = [c.value for c in next(worksheet.iter_rows(min_row=1, max_row=1))]
    total_col = (headers.index("total") + 1) if "total" in headers else None

    for row_idx, row in enumerate(worksheet.iter_rows(), start=1):
        is_header = row_idx == 1
        for col_idx, cell in enumerate(row, start=1):
            cell.alignment = Alignment(horizontal="left", vertical="center")
            cell.border = GRID
            if is_header:
                cell.fill = HEADER_FILL
                cell.font = FONT_HEADER
            elif col_idx == 1:
                cell.fill = DARK_BG
                cell.font = FONT_LIGHT
            elif total_col and col_idx == total_col:
                cell.fill = DARK_BG
                val = cell.value
                cell.font = FONT_RED_LIGHT if (val is not None and val != 0 and val != 0.0) else FONT_LIGHT
            else:
                cell.fill = NO_FILL
                val = cell.value
                cell.font = FONT_RED if (val is not None and val != 0 and val != 0.0) else FONT_DARK


def color_dtype_column(worksheet):
    for row in worksheet.iter_rows(min_row=2, min_col=2, max_col=2):
        for cell in row:
            dtype_str = str(cell.value).lower() if cell.value else ""
            for key, fill in DTYPE_FILLS.items():
                if key in dtype_str:
                    cell.fill = fill
                    break


def format_pct_columns(worksheet):
    headers = [cell.value for cell in next(worksheet.iter_rows(min_row=1, max_row=1))]
    for col_idx, header in enumerate(headers, start=1):
        if header != "column":
            for row in worksheet.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    cell.number_format = "0.00%"


In [62]:
output_path = "data_check_report.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for filename in sorted(os.listdir(Path)):
        if filename.endswith(".parquet"):
            filepath = os.path.join(Path, filename)
            name = os.path.splitext(filename)[0][:20]
            dtypes, blank_counts, blank_pcts, date_info = check_file(filepath)
            sheets = [
                (dtypes,       f"{name}_dtypes"),
                (blank_counts, f"{name}_null_count"),
                (blank_pcts,   f"{name}_null_%"),
                (date_info,    f"{name}_dates"),
            ]
            for df_out, sheet in sheets:
                df_out.to_excel(writer, sheet_name=sheet, index=False)
                ws = writer.sheets[sheet]
                autofit_columns(ws)
                apply_dark_theme(ws)
            color_dtype_column(writer.sheets[f"{name}_dtypes"])
            format_pct_columns(writer.sheets[f"{name}_null_%"])

print(f"Report saved to: {output_path}")


Report saved to: data_check_report.xlsx
